In [0]:
"""Functions shared by Databricks workspace notebooks."""
import json
import uuid
from datetime import datetime, timezone
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               DateType, TimestampType)
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG = 'github_health'
REPOS = ['apache/spark', 'pandas-dev/pandas', 'scikit-learn/scikit-learn']
ENTITIES = ['commits', 'issues', 'pulls']
ROOT = '/Volumes/github_health/raw/bronze_data/bronze'
LOG_TABLE = CATALOG + '.ops.pipeline_execution_logs'
WM_TABLE = CATALOG + '.ops.pipeline_watermarks'

LOG_SCHEMA = StructType([
    StructField('execution_id', StringType(), False), StructField('batch_id', StringType(), False),
    StructField('notebook_name', StringType()), StructField('layer', StringType()),
    StructField('run_date', DateType()), StructField('load_type', StringType()),
    StructField('start_time', TimestampType()), StructField('end_time', TimestampType()),
    StructField('status', StringType()), StructField('rows_processed', LongType()),
    StructField('error_message', StringType()),
    StructField('upserted_rows', LongType()), StructField('quarantined_rows', LongType())
])
WM_SCHEMA = StructType([
    StructField('repository', StringType(), False), StructField('entity', StringType(), False),
    StructField('layer', StringType(), False),
    StructField('last_successful_watermark', TimestampType()),
    StructField('batch_id', StringType()), StructField('updated_at', TimestampType())
])

def now():
    return datetime.now(timezone.utc).replace(tzinfo=None)

def parse_iso(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc).replace(tzinfo=None)

def iso(value):
    return value.replace(tzinfo=timezone.utc).isoformat().replace('+00:00', 'Z')

def log(spark, execution_id, batch_id, notebook, layer, run_date, load_type,
        start, status, rows, error=None, upserted=0, quarantined=0):
    from datetime import date
    spark.createDataFrame([(
        execution_id, batch_id, notebook, layer, date.fromisoformat(run_date),
        load_type, start, now(), status, int(rows), (error or '')[:30000] or None,
        int(upserted), int(quarantined)
    )], LOG_SCHEMA).write.format('delta').mode('append').saveAsTable(LOG_TABLE)

def merge_changed(spark, source, table, keys='record_key', change_column='payload_hash'):
    """Upsert only records with genuinely newer source timestamps.

    Uses source_updated_at on both source and target. Missing/equal timestamps
    do not overwrite existing records, preventing stale historical replay.
    """
    cols = source.columns
    # For existing rows update only when their business payload changed.
    condition = f't.{keys} = s.{keys}'
    changed = (
        f'NOT (t.{change_column} <=> s.{change_column}) '
        'AND s.source_updated_at IS NOT NULL '
        'AND (t.source_updated_at IS NULL OR s.source_updated_at > t.source_updated_at)'
    )
    update = {c: f's.`{c}`' for c in cols if c != keys}
    (DeltaTable.forName(spark, table).alias('t')
       .merge(source.alias('s'), condition)
       .whenMatchedUpdate(condition=changed, set=update)
       .whenNotMatchedInsert(values={c: f's.`{c}`' for c in cols})
       .execute())
    # These are the transaction's Delta MERGE operation metrics.
    metrics = spark.sql(f'DESCRIBE HISTORY {table} LIMIT 1').first()['operationMetrics']
    return (int(metrics.get('numTargetRowsInserted', 0)) +
            int(metrics.get('numTargetRowsUpdated', 0)))

def quarantine(spark, df, layer):
    """Idempotent quarantine MERGE keyed by deterministic quarantine_id."""
    table = f'{CATALOG}.ops.{layer}_quarantine'
    if not df.take(1):
        return 0
    df = df.dropDuplicates(['quarantine_id'])
    n = df.count()
    values = {c: f's.`{c}`' for c in df.columns}
    (DeltaTable.forName(spark, table).alias('t')
       .merge(df.alias('s'), 't.quarantine_id = s.quarantine_id')
       .whenNotMatchedInsert(values=values)
       .execute())
    return n
